# Phase 1 — Chargement & Nettoyage Open Food Facts

In [1]:
import sys
sys.path.append('..')

from src.data_pipeline import (
    _trouver_csv,
    detecter_colonnes_disponibles,
    charger_et_nettoyer,
    filtrer_dataset,
    exporter_csv,
    importer_mongodb,
    CSV_PROPRE
)

print('Imports OK')

Imports OK


## 1. Détection du fichier source

In [2]:
chemin_csv = _trouver_csv()
print(f'Fichier trouvé : {chemin_csv}')
print(f'Taille : {chemin_csv.stat().st_size / 1e9:.2f} Go')

Fichier trouvé : c:\Users\lilas\Desktop\Open Data\notebooks\..\fr.openfoodfacts.org.products.csv
Taille : 13.26 Go


## 2. Colonnes disponibles

In [3]:
colonnes = detecter_colonnes_disponibles(chemin_csv)

Colonnes trouvées   : 22/22


## 3. Chargement chunked + nettoyage

In [4]:
df = charger_et_nettoyer(chemin_csv, taille_chunk=50_000)

Colonnes trouvées   : 22/22
Chargement par chunks de 50,000 lignes...
  Chunk 91 traité — 4,532,980 lignes au total
Chargement terminé : 4,532,980 lignes, 22 colonnes


## 4. Filtres qualité

In [5]:
df_propre = filtrer_dataset(df, seuil_completude=0.3)
df_propre.head()

Filtre qualité : 1,373,352 produits conservés (supprimé 3,159,628)


,code,product_name,brands,categories,stores,ingredients_text,allergens,additives_n,additives_tags,nutriscore_score,...,pnns_groups_1,pnns_groups_2,completeness,energy-kcal_100g,fat_100g,saturated-fat_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g
12,7,granola Bio le Chocolaté,Mg ricarica,"Aliments et boissons à base de végétaux, Alime...",NaN,HONIG stillende Frauen nicht geeignet. D bestr...,NaN,0.0,NaN,4.0,...,Fruits and vegetables,Dried fruits,0.7625,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,8,NaN,Zuegg,Cornish clotted cream shortbread,NaN,"Sojaproteinisolat, Weizen - protein, Kaffee-Ex...",NaN,1.0,en:e955,6.0,...,unknown,unknown,0.6875,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,9,xytitol pastilles,xylimgxyling,"it:Gestione sovrappeso, it:obesità, Wrap",NaN,NaN,NaN,NaN,NaN,-11.0,...,Composite foods,Sandwiches,0.5750,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,13,Powdered peanut butter,Pbfit,"Snacks, Meals, Rice dishes, Risottos, Powder p...",TGV,"Water, Leptospermum Scoparium Mel (Manuka Hone...",NaN,0.0,NaN,3.0,...,Composite foods,One-dish meals,0.9750,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,15,Madeleines ChocoLait,Apple bandit,"Snacks, Snacks sucrés, Biscuits et gâteaux, Gâ...",NaN,"Farine de blé 27%, chocolat au lait 18% (sucre...",NaN,6.0,"en:e322,en:e322i,en:e331,en:e422,en:e500,en:e503",20.0,...,Sugary snacks,Biscuits and cakes,0.7875,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Aperçu rapide

In [6]:
print(f'Shape final         : {df_propre.shape}')
print(f'Produits uniques    : {df_propre["code"].nunique():,}')
print(f'Avec Nutri-Score    : {df_propre["nutriscore_grade"].notna().sum():,}')
print(f'Avec NOVA group     : {df_propre["nova_group"].notna().sum():,}')
print('\nTaux de NaN par colonne :')
print((df_propre.isnull().mean() * 100).round(1).sort_values(ascending=False))

Shape final         : (1373352, 22)
Produits uniques    : 1,373,317
Avec Nutri-Score    : 1,373,167
Avec NOVA group     : 850,040

Taux de NaN par colonne :
energy-kcal_100g      84.4
allergens             78.2
stores                75.7
additives_tags        63.2
fiber_100g            61.8
nova_group            38.1
ingredients_text      33.3
additives_n           33.3
saturated-fat_100g    31.4
sugars_100g           31.3
fat_100g              31.2
proteins_100g         31.1
salt_100g             31.1
brands                20.5
product_name           1.7
categories             0.1
code                   0.0
nutriscore_score       0.0
pnns_groups_2          0.0
pnns_groups_1          0.0
nutriscore_grade       0.0
completeness           0.0
dtype: float64


## 6. Export CSV propre

In [7]:
exporter_csv(df_propre, CSV_PROPRE)

CSV exporté : c:\Users\lilas\Desktop\Open Data\notebooks\..\data\processed\openfoodfacts_clean.csv


## 7. Import MongoDB (nécessite .env configuré)

In [ ]:
importer_mongodb(df_propre)

c:\Users\lilas\anaconda3\Lib\site-packages\pymongo\pyopenssl_context.py:352: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  _crypto.X509.from_cryptography(x509.load_der_x509_certificate(cert))


ServerSelectionTimeoutError: ac-xzyex1a-shard-00-01.g5ukql5.mongodb.net:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),ac-xzyex1a-shard-00-00.g5ukql5.mongodb.net:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),ac-xzyex1a-shard-00-02.g5ukql5.mongodb.net:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6a17912989b7aef1e4bf2f2d, topology_type: Unknown, servers: [<ServerDescription ('ac-xzyex1a-shard-00-00.g5ukql5.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('ac-xzyex1a-shard-00-00.g5ukql5.mongodb.net:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>, <ServerDescription ('ac-xzyex1a-shard-00-01.g5ukql5.mongodb.net', 2701

: 